In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import os, warnings 
warnings.filterwarnings('ignore') 

from glob import glob
from setup import *
import stan

dpi = 300
fontsize = 8
figsize = 2.5

# dpi = 300
# fontsize = 10
# figsize = 2

# High-resolution figures for the manuscript

In [2]:
sample_list = []
for x in glob('data/Ravi_processed/gbm_qc/*'):
    sample_list.append(x.split(r'/')[-1][:-5])
sample_list.sort()

In [3]:
sample_list = ['UKF248', 'UKF260', 'UKF275', 'UKF313']
adatas = dict() # input of STAN
for sample in sample_list:
    adatas[sample] = sc.read_h5ad(f"data/Ravi_processed/gbm_qc/{sample}.h5ad")
    sc.pp.normalize_total(adatas[sample])
    sc.pp.log1p(adatas[sample])
    
adatas_tfa = dict()
for sample in sample_list:
    adata = sc.read_h5ad(f"results/ravi_tfa/{sample}.h5ad")
    adatas_tfa[sample] = stan.get_activity(adata, key='tfa_stan')
    
adatas_pwa = dict()
for sample in sample_list:
    adata = sc.read_h5ad(f"results/ravi_pwa/{sample}.h5ad")
    adatas_pwa[sample] = stan.get_activity(adata, key='pwa_stan')

adatas_d2c = dict()
for sample in sample_list:
    adatas_d2c[sample] = sc.read_h5ad(f"results/ravi_d2c/{sample}.h5ad")

## Sample information

In [ ]:
def plot_stain(adata, sample, path='plots_website/he_stain'):
    fig, ax = plt.subplots(1, 1, figsize=(figsize, figsize), dpi=dpi)
    plt.rc('font', size=fontsize) 
    sc.pl.spatial(adata, ax=ax, show=False, title=sample)

    ax.set_xlabel("")
    ax.set_ylabel("")
    plt.tight_layout(pad=1)
    plt.savefig(f'{path}/{sample}.pdf')
    # plt.close()
    return

In [ ]:
for sample in sample_list:
    path = f'plots_manuscript/{sample}'
    if not os.path.exists(path):
        os.mkdir(path)
    plot_stain(adatas[sample], sample, path)

In [ ]:
def plot_metaprogram(adata, sample, path='plots_website/metaprogram'):
    fig, ax = plt.subplots(1, 1, figsize=(figsize*1.6, figsize), dpi=dpi)
    plt.rc('font', size=8) 
    sc.pl.spatial(adata, color='annotation', palette=metaprogram_palette, alpha_img=0, size=1.8, 
                  ax=ax, show=False, title='Metaprogram', legend_fontsize=fontsize*0.8, na_in_legend=False)
    ax.set_xlabel("")
    ax.set_ylabel("")
    plt.tight_layout(pad=1)
    plt.savefig(f'{path}/metaprogram.pdf')
    return

In [ ]:
for sample in sample_list:
    path = f'plots_manuscript/{sample}'
    if not os.path.exists(path):
        os.mkdir(path)
    plot_metaprogram(adatas[sample], sample, path)

## mRNA plots

In [5]:
def plot_spatial_expression(adata, sample, gene, vmax, path='plots_website/mrna'):
    title = f"{gene} expression\nMoran's I = {moran_mran_df.loc[gene, sample]:.2f}"
    fig, ax = plt.subplots(1, 1, figsize=(figsize, figsize), dpi=dpi)
    plt.rc('font', size=fontsize) 
    sc.pl.spatial(adata, color=gene, alpha_img=0, ax=ax, show=False, cmap="viridis", 
                  legend_fontsize=fontsize, size=1.8,
                  vmin=adata.X.min(), vmax=vmax
                 )
    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=fontsize)
    ax.set_title(title, fontsize=fontsize)
    ax.set_xlabel("")
    ax.set_ylabel("")

    plt.tight_layout(pad=1)
    
    # Save the plot
    plt.savefig(f'{path}/mrna_{gene}.pdf')
    plt.close()

In [39]:
# fname = 'dataframes/morani_ravi/mrna_lr_per_sample.csv'
# moran_mran_df = pd.read_csv(fname, index_col=0)

marker_genes = ['VIM', 'CD44', 'ADM']
adatas_markers = dict()
for sample in ['UKF260']:
    adata = adatas[sample]
    adatas_markers[sample] = adata[:, [gene for gene in marker_genes if gene in adata.var_names]]
moran_mran_df = stan.compute_morans_i(adatas_markers)

  0%|          | 0/100 [00:00<?, ?/s]

In [40]:
sample = 'UKF260'
adata = adatas[sample]
adata.to_df()[['VIM', 'CD44', 'ADM']].max()

VIM     4.012707
CD44    2.693678
ADM     2.978158
dtype: float32

In [41]:
for gene in ['VIM', 'CD44', 'ADM']: 
    path = f'plots_manuscript/{sample}'
    if not os.path.exists(path):
        os.mkdir(path)
    plot_spatial_expression(adata, sample, gene, vmax=4, path=path)

In [19]:
sample = 'UKF275'
adata = adatas[sample]
adata.to_df()[marker_genes].max()

S100A10    3.545844
dtype: float32

In [20]:
for gene in marker_genes: ## remove [:3]
    path = f'plots_manuscript/{sample}'
    if not os.path.exists(path):
        os.mkdir(path)
    plot_spatial_expression(adata, sample, gene, vmax=4, path=path)

In [22]:
sample = 'UKF313'
adata = adatas[sample]
adata.to_df()[['CD74', 'B2M', 'CD44']].max()

CD74    3.066537
B2M     3.549026
CD44    2.697131
dtype: float32

In [23]:
for gene in ['CD74', 'B2M', 'CD44']: ## remove [:3]
    path = f'plots_manuscript/{sample}'
    if not os.path.exists(path):
        os.mkdir(path)
    plot_spatial_expression(adata, sample, gene, vmax=3.5, path=path)

## TF plots

In [30]:
def plot_spatial_activity(adata, sample, tf, path='plots_website/transcription'):
    title = f"{tf} activity\nMoran's I = {moran_tfa_df.loc[tf, sample]:.2f}"
    fig, ax = plt.subplots(1, 1, figsize=(figsize, figsize), dpi=dpi)
    plt.rc('font', size=fontsize) 
    sc.pl.spatial(adata, color=tf, alpha_img=0, ax=ax, show=False, cmap="plasma", 
                  legend_fontsize=fontsize, size=1.8,)
    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=fontsize)
    ax.set_title(title, fontsize=fontsize)
    ax.set_xlabel("")
    ax.set_ylabel("")
    plt.tight_layout(pad=1)
    
    # Save the plot
    plt.savefig(f'{path}/tfa_{tf}.pdf')
    plt.close()

In [33]:
fname = 'dataframes/morani_ravi/tfa_per_sample.csv'
moran_tfa_df = pd.read_csv(fname, index_col=0)

In [42]:
for tf in ['SOX2', 'HIF1A', 'PAX6']: ## remove [:3]
    for sample in ['UKF260', 'UKF275']:
        path = f'plots_manuscript/{sample}'
        if not os.path.exists(path):
            os.mkdir(path)
        plot_spatial_activity(adatas_tfa[sample], sample, tf, path=path)

In [35]:
for tf in ['STAT3', 'SNAI2']: ## remove [:3]
    for sample in ['UKF313']:
        path = f'plots_manuscript/{sample}'
        if not os.path.exists(path):
            os.mkdir(path)
        plot_spatial_activity(adatas_tfa[sample], sample, tf, path=path)

## Pathway plots

In [27]:
def plot_spatial_activity_pdf(adata, sample, pathway, path='plots_website/pathway'):
    title = f"{hallmarks[pathway]} activity\nMoran's I = {moran_pwa_df.loc[pathway, sample]:.2f}"
    fig, ax = plt.subplots(1, 1, figsize=(figsize, figsize), dpi=dpi)
    plt.rc('font', size=fontsize) 
    sc.pl.spatial(adata, color=pathway, alpha_img=0, ax=ax, show=False, cmap="rocket", 
                  legend_fontsize=fontsize, size=1.8)
    ax.set_title(title, fontsize=fontsize)
    ax.set_xlabel("")
    ax.set_ylabel("")
    plt.tight_layout(pad=1)
    plt.savefig(f'{path}/pwa_{pathway}.pdf')
    plt.close()

In [25]:
fname = 'dataframes/morani_ravi/pwa_per_sample.csv'
moran_pwa_df = pd.read_csv(fname, index_col=0)

In [ ]:
for sample in ['UKF260']:
    adata = adatas_pwa[sample]
    for pathway in ['HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION']:
        path = f'plots_manuscript/{sample}'
        if not os.path.exists(path):
            os.mkdir(path)
        plot_spatial_activity_pdf(adata, sample, pathway, path)

In [ ]:
for sample in ['UKF275']:
    adata = adatas_pwa[sample]
    for pathway in ['HALLMARK_HYPOXIA']:
        path = f'plots_manuscript/{sample}'
        if not os.path.exists(path):
            os.mkdir(path)
        plot_spatial_activity_pdf(adata, sample, pathway, path)

In [29]:
for sample in ['UKF313']:
    adata = adatas_pwa[sample]
    for pathway in ['HALLMARK_TNFA_SIGNALING_VIA_NFKB', 'HALLMARK_IL6_JAK_STAT3_SIGNALING', 'HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION']:
        path = f'plots_manuscript/{sample}'
        if not os.path.exists(path):
            os.mkdir(path)
        plot_spatial_activity_pdf(adata, sample, pathway, path)

## Drug plots

In [ ]:
def plot_spatial_score_pdf(adata, sample, drug, path='plots_website/drug'):
    d1, d2 = drug.split('|')
    d3 = split_at_natural_break(d2)
    title = f"{d1} {d3}\nMoran's I = {moran_d2c_df.loc[drug, sample]:.2f}"
    
    fig, ax = plt.subplots(1, 1, figsize=(figsize, figsize), dpi=dpi)
    plt.rc('font', size=fontsize) 
    sc.pl.spatial(adata, color=drug, alpha_img=0, ax=ax, show=False, cmap="inferno", 
                  legend_fontsize=fontsize, size=1.8)
    ax.set_title(title, fontsize=fontsize)
    ax.set_xlabel("")
    ax.set_ylabel("")
    plt.tight_layout(pad=1)
    plt.savefig(f'{path}/d2c_{d1}.pdf')
    plt.close()

In [ ]:
marker_genes = ['CHEMBL1201825|RANIBIZUMAB', 'CHEMBL1201421|PEGAPTANIB SODIUM',
               'CHEMBL1201583|BEVACIZUMAB', 'CHEMBL3707357|BROLUCIZUMAB']
adatas_markers = dict()
for sample in sample_list:
    adata = adatas_d2c[sample]
    adatas_markers[sample] = adata[:, [gene for gene in marker_genes if gene in adata.var_names]]
moran_d2c_df = stan.compute_morans_i(adatas_markers)

In [ ]:
# fontsize = 7.5
for sample in ['UKF260']: ## remove [:3]
    adata = adatas_d2c[sample]
    for drug in ['CHEMBL1201825|RANIBIZUMAB', 'CHEMBL1201421|PEGAPTANIB SODIUM']: ## remove [:3]
        path = f'plots_manuscript/{sample}'
        if not os.path.exists(path):
            os.mkdir(path)
        plot_spatial_score_pdf(adata, sample, drug, path)

In [ ]:
for sample in ['UKF275']: ## remove [:3]
    adata = adatas_d2c[sample]
    for drug in ['CHEMBL1201583|BEVACIZUMAB', 'CHEMBL3707357|BROLUCIZUMAB']: ## remove [:3]
        path = f'plots_manuscript/{sample}'
        if not os.path.exists(path):
            os.mkdir(path)
        plot_spatial_score_pdf(adata, sample, drug, path)

## Additional Samples

In [ ]:
fontsize = 10
fig, axes = plt.subplots(2, 7, figsize=(14, 4), dpi=dpi)
plt.rc('font', size=fontsize)

# 定义参数
samples = ['UKF269', 'UKF304']
genes = ['SOX2', 'SOX10', 'SMAD4']

for i, sample in enumerate(samples):
    adata = adatas[sample]
    adata_tf = adatas_tfa[sample]
    # 第一列：H&E stain
    sc.pl.spatial(adata, ax=axes[i,0], show=False, title=f"{sample}")
    
    for j, gene in enumerate(genes):
        # Gene expression (columns 1,3,5)
        col_gene = 1 + j*2
        title = f"{gene} expression\nMoran's I = {moran_mran_df.loc[gene, sample]:.2f}"
        sc.pl.spatial(adata, color=gene, alpha_img=0, ax=axes[i, col_gene], show=False, 
                      cmap="viridis", legend_fontsize=fontsize, size=1.8)
        cbar = axes[i, col_gene].collections[0].colorbar
        cbar.ax.tick_params(labelsize=fontsize)
        axes[i, col_gene].set_title(title, fontsize=fontsize)
        
        # TF activity (columns 2,4,6)
        col_tf = 2 + j*2
        title = f"{gene} activity\nMoran's I = {moran_tfa_df.loc[gene, sample]:.2f}"
        sc.pl.spatial(adata_tf, color=gene, alpha_img=0, ax=axes[i, col_tf], show=False, 
                      cmap="plasma", legend_fontsize=fontsize, size=1.8)
        cbar = axes[i, col_tf].collections[0].colorbar
        cbar.ax.tick_params(labelsize=fontsize)
        axes[i, col_tf].set_title(title, fontsize=fontsize)

# 修改bug：遍历axes的所有子图
for ax_row in axes:
    for ax in ax_row:
        ax.set_xlabel("")
        ax.set_ylabel("")
        
plt.tight_layout(pad=1)
plt.savefig('plots_manuscript/spatial_plots.pdf')
plt.close()

In [ ]:
fontsize = 10
fig, axes = plt.subplots(1, 5, figsize=(10, 2), dpi=dpi)
plt.rc('font', size=fontsize)

sample = 'UKF248'
adata = adatas[sample]
adata_tf = adatas_tfa[sample]
sc.pl.spatial(adata, ax=axes[0], show=False, title=f"{sample}")

genes = ['SOX2', 'PAX6']
for j, gene in enumerate(genes):
    # Gene expression (columns 1,3,5)
    col_gene = 1 + j*2
    title = f"{gene} expression\nMoran's I = {moran_mran_df.loc[gene, sample]:.2f}"
    sc.pl.spatial(adata, color=gene, alpha_img=0, ax=axes[col_gene], show=False, 
                  cmap="viridis", legend_fontsize=fontsize, size=1.8)
    cbar = axes[col_gene].collections[0].colorbar
    cbar.ax.tick_params(labelsize=fontsize)
    axes[col_gene].set_title(title, fontsize=fontsize)

    # TF activity (columns 2,4,6)
    col_tf = 2 + j*2
    title = f"{gene} activity\nMoran's I = {moran_tfa_df.loc[gene, sample]:.2f}"
    sc.pl.spatial(adata_tf, color=gene, alpha_img=0, ax=axes[col_tf], show=False, 
                  cmap="plasma", legend_fontsize=fontsize, size=1.8)
    cbar = axes[col_tf].collections[0].colorbar
    cbar.ax.tick_params(labelsize=fontsize)
    axes[col_tf].set_title(title, fontsize=fontsize)

# 修改bug：遍历axes的所有子图
for ax in axes:
    ax.set_xlabel("")
    ax.set_ylabel("")
        
plt.tight_layout(pad=1)
plt.savefig('plots_manuscript/spatial_plots.pdf')
plt.close()

In [ ]:
fontsize = 10
sample = 'UKF269'
adata = adatas[sample]
adata.to_df()[['SOX2', 'SOX10', 'SMAD4', 'FOS', 'JUN', 'ATF3']].max()

In [ ]:
for gene in ['SOX2', 'SOX10', 'SMAD4', 'FOS', 'JUN', 'ATF3']: 
    path = f'plots_manuscript/{sample}'
    if not os.path.exists(path):
        os.mkdir(path)
    plot_spatial_expression(adata, sample, gene, vmax=2.5, path=path)

In [ ]:
for tf in ['SOX2', 'SOX10', 'SMAD4', 'FOS', 'JUN', 'ATF3']: 
    for sample in ['UKF269']:
        path = f'plots_manuscript/{sample}'
        if not os.path.exists(path):
            os.mkdir(path)
        plot_spatial_activity(adatas_tfa[sample], sample, tf, path=path)